# Notebook 3: Ollama Integration and Model Serving

Focus:
- Local inference architecture
- Prompting + request payloads
- Latency and resource behavior


## What is Ollama?

### Definition
Ollama is local model runtime exposing HTTP API for generation, chat, embeddings, and multimodal tasks.

### Theory
It wraps model lifecycle (download, load, infer) behind stable endpoints.

### Motivation
Local inference improves privacy and cost control.

### Real-World Examples
Teams can build internal copilots without sending data to external APIs.

### Visual Explanation
```
App -> http://localhost:11434/api/* -> Local Model Runtime
```

### Code Explanation
`src/ollama_client.py` implements thin, testable API client with normalized error handling.

### Best Practices
Use retries and explicit timeouts for local API calls too.

### Common Mistakes
Hardcoding assumptions about endpoint availability.


In [ ]:
from src.ollama_client import OllamaClient

client = OllamaClient()
models = client.list_models()
client.close()
models[:10]

## Prompt Lifecycle and Structured Output

### Definition
Prompt lifecycle includes system instruction, user context, and output schema constraints.

### Theory
Structured output reduces postprocessing ambiguity and improves UI reliability.

### Motivation
Interviewers expect engineers to explain why prompt format impacts parse reliability.

### Real-World Examples
Sentiment tab requests JSON output with label/confidence/explanation fields.

### Visual Explanation
```
System Prompt + User Input -> Model -> JSON Parse -> Typed Result
```

### Code Explanation
`SentimentAnalyzer` and `Summarizer` parse JSON first then apply safe fallback.

### Best Practices
Keep parse fallbacks deterministic and logged.

### Common Mistakes
Blindly trusting model returns valid JSON 100% of time.


In [ ]:
from src.sentiment import SentimentAnalyzer

analyzer = SentimentAnalyzer()
result = analyzer.analyze('This workflow is surprisingly smooth and useful!')
analyzer.close()
result.model_dump()

In [ ]:
from src.translation import Translator

translator = Translator()
translation = translator.translate('Machine learning systems need observability.', 'English', 'Spanish')
translator.close()
translation.model_dump()

## Resource Requirements and Optimization

### Definition
Large models increase latency and memory usage; careful model-task mapping matters.

### Theory
Lazy loading and warmup reduce user-facing cold-start delays.

### Motivation
Optimization decisions directly impact UX and hardware cost.

### Real-World Examples
Small model for sentiment, larger model for chat reasoning, OCR-specialized model for documents.

### Visual Explanation
```
Task complexity ↑ -> model size/latency ↑
```

### Code Explanation
App includes warmup controls and benchmark tab for evidence-driven model selection.

### Best Practices
Track latency p95 and memory, not only average latency.

### Common Mistakes
Using one large model for every task without benchmarking.
